### A set of Script Tools:
Sharing and accessing data is a common friction point in GIS workflows. Let's turn the processes we just went through (pulling data from AGOL, pulling data from an API, and pushing data to AGOL) into a set of four Script Tools, then chain some of them together into a model. ArcPro lets you bundle scripts inside of Toolsets.

- Step 1: Right click on the PythonIsCool.atbx and then New -> Toolset.
- Give the Toolset a descriptive name: `AGOLDataSharingWorkflow`
- Let's build the first tool in the set!

#### Script Tool 1: FindAGOLItem
This will be a script tool, so we will need to set parameters in ArcPro to make it interactive. Use the following parameters:

Parameters (set in the tool’s Parameters tab, in this order):
	Label	- Data type	- Required	- Default / filter

0)	Search terms	- String	- No	
1)	Search my organization only	- Boolean	- No	- Checked
2)	Owner	- String	- No	
3)	Item type	- String	- No	- Value List: Any, CSV, Feature Service, Microsoft Excel, Shapefile, File Geodatabase, Web Map, Code Sample; default Any
4)	Modified on or after	- Date	- No	
5)	Modified on or before	- Date	- No	

**IMPORTANT**: When you create the script tool, you should make sure that the script (below) is imported AND embedded. This way you can share the toolbox later and it will run as it is.

In [ ]:
import arcpy
from datetime import datetime, timedelta
from arcgis.gis import GIS

MAX_SHOWN = 25     # how many matches to list
FETCH_LIMIT = 200  # how many matches to ask AGOL for before we apply the date filter


def connect(require_signin):
    """Signed in through Pro if possible; otherwise anonymous (public items only)."""
    try:
        gis = GIS("pro")
        arcpy.AddMessage(f"Signed in as {gis.users.me.username}")
        return gis
    except Exception:
        if require_signin:
            raise RuntimeError(
                "You are not signed in to ArcGIS Pro. Searching your organization needs a sign-in. "
                "Sign in (top right of Pro), then run the tool again. "
                "Or uncheck 'Search my organization only' to search public items without signing in."
            )
        arcpy.AddWarning("Not signed in, so only PUBLIC items can be searched.")
        return GIS()


def modified_dt(item):
    # AGOL stores dates as milliseconds since 1970; convert to a normal date
    return datetime.fromtimestamp(item.modified / 1000)


def main():
    terms     = arcpy.GetParameterAsText(0).strip()
    org_only  = arcpy.GetParameter(1)
    owner     = arcpy.GetParameterAsText(2).strip()
    item_type = arcpy.GetParameterAsText(3).strip()
    after     = arcpy.GetParameter(4)
    before    = arcpy.GetParameter(5)

    # An empty optional Date may come back as None or an empty string; keep only real dates
    after  = after  if isinstance(after, datetime) else None
    before = before if isinstance(before, datetime) else None
    if item_type in ("", "Any"):
        item_type = None

    # Filters alone (just a type or date) would match thousands of items
    if not terms and not owner:
        arcpy.AddError("Enter search terms and/or an owner. Type and date filters can't be used on their own.")
        return

    gis = connect(require_signin=bool(org_only))

    # Build the AGOL search string, e.g.  fire hydrants owner:jsmith
    query = " ".join(p for p in [terms, f"owner:{owner}" if owner else ""] if p)
    arcpy.AddMessage(f"Searching for: {query}  |  type: {item_type or 'any'}  |  "
                     f"{'my organization' if org_only else 'public items'}")

    results = gis.content.search(
        query=query,
        item_type=item_type,
        sort_field="modified",      # newest first
        sort_order="desc",
        max_items=FETCH_LIMIT,
        outside_org=not org_only,   # False = my org only, True = public items beyond my org
    )

    # Date filtering is done here in Python so the logic is visible
    if after:
        results = [r for r in results if modified_dt(r) >= after]
    if before:
        results = [r for r in results if modified_dt(r) < before + timedelta(days=1)]  # include that whole day

    if not results:
        arcpy.AddWarning(
            "No matches found. Try: fewer or simpler words (one word like 'hydrant' often works better than "
            "a phrase), removing the date, owner or type filter, or unchecking 'Search my organization only' "
            "if the data might live outside your org. If it's someone else's private item, it must be "
            "shared with you."
        )
        return

    shown = results[:MAX_SHOWN]
    arcpy.AddMessage(f"\nFound {len(results)} match(es). Showing {len(shown)}, newest first:\n")
    for i, r in enumerate(shown, 1):
        arcpy.AddMessage(f"{i}. {r.title}\n"
                         f"     Type: {r.type}  |  Owner: {r.owner}  |  Modified: {modified_dt(r):%Y-%m-%d}\n"
                         f"     Item ID: {r.id}")
    if len(results) > len(shown):
        arcpy.AddMessage(f"\n...and {len(results) - len(shown)} more. Add terms or filters to narrow it down.")
    arcpy.AddMessage("\nNext: copy the Item ID(s) you want into the 'Add AGOL Items to Map' tool. "
                     "Not what you wanted? Re-run this tool with different terms.")


try:
    main()
except Exception as e:
    arcpy.AddError(str(e))

How it works:

* arcpy.AddMessage, AddWarning and AddError are three severity levels in the Geoprocessing pane. A warning (“no matches”) lets the tool finish with a yellow note. An error turns the run red.
* GetParameter returns real Python objects (bool, datetime), while GetParameterAsText always returns text. That’s why the dates use GetParameter: it avoids locale problems like 3/4/2026 meaning March 4 or April 3.
* The search returns the newest 200 matches, and the date filter runs on those. With very common terms and a date far in the past, matches could fall outside the 200. That’s fine for a workshop, but worth knowing.
* Search matches whole words (with some stemming, so “hydrants” finds “hydrant”). It won’t match firehydrants as one word. This is why the no-match message suggests trying a single word.

#### Script Tool 2: AddAGOLItemToMap
Parameters (set in the tool’s Parameters tab, in this order):
	Label	- Data type	- Required	- Default

0)	Item IDs	- String	- Yes	
1)	Download folder (for items like csv files)	- Folder	- No	- blank; falls back to project's home folder
2)	Add to active map	- Boolean	- No    - Checked	


In [ ]:
import re
import arcpy
from arcgis.gis import GIS

SERVICE_TYPES = ("Feature Service", "Map Service")


def connect():
    """Signed in through Pro if possible; otherwise anonymous (can still fetch PUBLIC items)."""
    try:
        gis = GIS("pro")
        arcpy.AddMessage(f"Signed in as {gis.users.me.username}")
        return gis
    except Exception:
        arcpy.AddWarning("Not signed in to ArcGIS Pro: only PUBLIC items can be retrieved.")
        return GIS()


def add_to_map(m, path_or_url, label):
    try:
        m.addDataFromPath(path_or_url)
        arcpy.AddMessage(f"   Added to map: {label}")
    except Exception as e:
        arcpy.AddWarning(f"   Could not add '{label}' to the map automatically ({e}). "
                         f"You can add it by hand with Add Data > Add Data From Path: {path_or_url}")


def main():
    raw      = arcpy.GetParameterAsText(0)
    folder   = arcpy.GetParameterAsText(1)
    add_flag = arcpy.GetParameter(2)

    # Pull every 32-character ID out of whatever was pasted (a URL, a list, extra text)
    # dict.fromkeys removes duplicates while keeping the order
    ids = list(dict.fromkeys(re.findall(r"[0-9a-fA-F]{32}", raw)))
    if not ids:
        arcpy.AddError("No item ID found. An item ID is 32 letters/numbers, found in an item's URL after 'id='.")
        return

    project = arcpy.mp.ArcGISProject("CURRENT")
    if not folder:
        folder = project.homeFolder
        arcpy.AddMessage(f"No download folder chosen; file items will go to {folder}")

    m = project.activeMap if add_flag else None
    if add_flag and m is None:
        arcpy.AddWarning("No active map. Click into a map view and re-run to add items; file items will still download.")

    gis = connect()
    done, failed = 0, 0

    for item_id in ids:
        arcpy.AddMessage(f"\nItem {item_id}")
        try:
            item = gis.content.get(item_id)   # returns None if it doesn't exist or isn't visible to you
        except Exception:
            item = None
        if item is None:
            arcpy.AddError(f"   No item found with ID {item_id}. Check for typos, and that it's public "
                           f"(or shared with you while you're signed in).")
            failed += 1
            continue
        arcpy.AddMessage(f"   Found: {item.title} ({item.type}) by {item.owner}")

        if item.type in SERVICE_TYPES:
            # Hosted layers aren't files: we add them to the map by their web address
            sublayers = list(item.layers or []) + list(getattr(item, "tables", None) or [])
            if not sublayers:
                arcpy.AddWarning("   This service has no layers to add.")
            elif m is None:
                arcpy.AddWarning(f"   Not added (no active map, or box unchecked). Layer URL: {sublayers[0].url}")
            else:
                for lyr in sublayers:
                    add_to_map(m, lyr.url, lyr.properties.name)
            done += 1

        elif item.type == "Web Map":
            arcpy.AddWarning("   Web maps can't be added with this tool. Open it from Catalog > Portal instead.")
            failed += 1

        else:
            # File items (CSV, zip, Excel...): download, then add if it's a CSV
            try:
                path = item.download(save_path=folder)
            except Exception as e:
                arcpy.AddError(f"   Could not download '{item.title}': {e}. The owner may not allow "
                               f"downloads, or it isn't shared with you.")
                failed += 1
                continue
            arcpy.AddMessage(f"   Downloaded to: {path}")
            if item.type == "CSV" and m is not None:
                add_to_map(m, path, item.title)
            elif item.type != "CSV":
                arcpy.AddMessage("   This file type isn't added automatically. Unzip or open it from the folder above.")
            done += 1

    arcpy.AddMessage(f"\nFinished: {done} item(s) retrieved, {failed} problem(s).")


try:
    main()
except Exception as e:
    arcpy.AddError(str(e))

How it works:

* re.findall(r"[0-9a-fA-F]{32}", raw) means “find every run of exactly 32 hex characters.” That’s how pasting a whole URL works.
* Services are added by URL rather than downloaded. `addDataFromPath` adds a layer to a map from a local path or a URL. A CSV item is a file, so the tool downloads it first and then adds the local path. 
* The type branching exists because AGOL items are different kinds of things. A hosted feature layer is a live service, a CSV is a file, and a web map is neither. Each needs its own handling.
* With no sign-in the tool falls back to the anonymous connection, which can fetch public items.

#### Script Tool 3: FetchINatObservations
This tool pulls observations from the iNaturalist API (like Notebook 5) and saves them as a point feature class. Use the following parameters:

Parameters (set in the tool's Parameters tab, in this order):
	Label	- Data type	- Required	- Default

0)	Scientific name	- String	- Yes	
1)	Place ID	- Long	- No	- 9 (New Mexico)
2)	Number of observations (max 200)	- Long	- No	- 200
3)	Output feature class	- Feature Class	- Yes	- Direction: **Output**

The output parameter matters. It is what lets Model Builder hand this tool's result to the next tool.

In [ ]:
import os
import arcpy
import pandas as pd
import requests

URL = "https://api.inaturalist.org/v1/observations"


def main():
    taxon_name = arcpy.GetParameterAsText(0).strip()
    place_id   = arcpy.GetParameter(1)
    per_page   = arcpy.GetParameter(2)
    out_fc     = arcpy.GetParameterAsText(3)

    # Optional numbers can come back empty; fall back to the workshop defaults
    place_id = int(place_id) if place_id else 9
    per_page = min(int(per_page), 200) if per_page else 200   # iNaturalist allows at most 200 per page

    arcpy.AddMessage(f"Asking iNaturalist for up to {per_page} observations of {taxon_name} in place {place_id}...")
    params = {
        "taxon_name": taxon_name,
        "place_id": place_id,
        "per_page": per_page,
        "quality_grade": "research",
    }
    response = requests.get(URL, params=params, timeout=30)
    if response.status_code != 200:
        arcpy.AddError(f"iNaturalist returned status {response.status_code}. "
                       f"If it is 429, wait a minute and run the tool again.")
        return

    records = []
    for obs in response.json().get("results", []):
        location = obs.get("location")
        if not location:                      # skip records without coordinates
            continue
        lat, lon = location.split(",")
        taxon = obs.get("taxon") or {}        # the taxon block holds the species names
        records.append({
            "id": obs.get("id"),
            "observed_on": obs.get("observed_on"),
            "time_zone": obs.get("time_zone"),
            "latitude": float(lat),
            "longitude": float(lon),
            "scientific_name": taxon.get("name"),
            "common_name": taxon.get("preferred_common_name") or obs.get("species_guess"),
        })

    if not records:
        arcpy.AddError("No observations with locations came back. Check the scientific name and place ID.")
        return

    # Same cleaning and column order as Notebook 3
    df = pd.DataFrame(records)
    df["date_dt"] = pd.to_datetime(df["observed_on"], errors="coerce")
    df["obs_year"] = df["date_dt"].dt.year
    df = df[["id", "date_dt", "time_zone", "latitude", "longitude",
             "scientific_name", "common_name", "obs_year"]]

    # Write a temporary CSV, then turn it into points
    csv_path = os.path.join(arcpy.env.scratchFolder, "inat_fetch.csv")
    df.to_csv(csv_path, index=False)
    arcpy.AddMessage(f"Fetched {len(df)} observations.")

    arcpy.env.overwriteOutput = True          # so the tool can be rerun with the same output name
    arcpy.management.XYTableToPoint(
        in_table=csv_path,
        out_feature_class=out_fc,
        x_field="longitude",
        y_field="latitude",
        coordinate_system=arcpy.SpatialReference(4326)
    )
    arcpy.AddMessage(f"Created feature class: {out_fc}")


try:
    main()
except Exception as e:
    arcpy.AddError(str(e))

How it works:

* This is Notebook 5's black bear fetch, Notebook 3's cleaning steps, and the XYtoPoint tool combined into one tool. Every piece of it is code you have already seen.
* `scratchFolder` is a temporary folder that ArcPro provides for scripts. The CSV is only a stepping stone, so it goes there and not in your data folders.
* A response of 429 means iNaturalist is asking you to slow down. Waiting a minute and running again fixes it.

#### Script Tool 4: UploadFeatureClassToAGOL
This tool exports a feature class's table to a CSV and uploads it to your AGOL content, like the CSV upload in Notebook 5. Uploading a CSV doesn't need publisher privileges. Use the following parameters:

Parameters (set in the tool's Parameters tab, in this order):
	Label	- Data type	- Required	- Default

0)	Input feature class	- Feature Class	- Yes	
1)	Item title	- String	- Yes	
2)	Tags (separate with commas)	- String	- No	- Python, ArcGIS Pro

In [ ]:
import os
import re
import arcpy
import pandas as pd
from arcgis.gis import GIS


def main():
    in_fc = arcpy.GetParameterAsText(0)
    title = arcpy.GetParameterAsText(1).strip()
    tags  = [t.strip() for t in arcpy.GetParameterAsText(2).split(",") if t.strip()]

    # Uploading needs a sign-in; an anonymous connection has no user attached
    try:
        gis = GIS("pro")
    except Exception:
        gis = None
    if gis is None or gis.users.me is None:
        arcpy.AddError("You are not signed in to ArcGIS Pro. Sign in (top right of Pro), then run the tool again.")
        return
    arcpy.AddMessage(f"Signed in as {gis.users.me.username}")

    # Read the attribute table (everything except geometry) into a dataframe
    fields = [f.name for f in arcpy.ListFields(in_fc) if f.type not in ("Geometry", "OID", "Blob", "Raster")]
    with arcpy.da.SearchCursor(in_fc, fields) as cursor:
        df = pd.DataFrame([list(row) for row in cursor], columns=fields)
    arcpy.AddMessage(f"Read {len(df)} rows and {len(fields)} columns from the feature class.")

    # Save a temporary CSV named after the title (letters and numbers only)
    safe_name = re.sub(r"\W+", "_", title).strip("_") or "upload"
    csv_path = os.path.join(arcpy.env.scratchFolder, f"{safe_name}.csv")
    df.to_csv(csv_path, index=False)

    item_properties = {
        "title": title,
        "type": "CSV",
        "tags": tags or ["Python"],
        "snippet": f"CSV with {len(df)} rows uploaded from ArcGIS Pro with a script tool.",
    }
    item = gis.content.add(item_properties, data=csv_path)
    arcpy.AddMessage(f"Uploaded: {item.title}\nItem ID: {item.id}")
    arcpy.AddMessage("You can paste that Item ID into the 'Add AGOL Items to Map' tool to bring it back.")


try:
    main()
except Exception as e:
    arcpy.AddError(str(e))

How it works:

* Notebook 5 uploaded a CSV that already existed. Here the CSV is created from the feature class first, then uploaded.
* `gis.users.me` is empty (None) for an anonymous connection, which is how the tool knows you are not really signed in.
* The tool prints the new Item ID, which is exactly what Script 2 needs. Together the tools make a loop: fetch, upload, find, and add back to a map.

### Chaining Tools with Model Builder
Esri's built-in tools cover a lot, but not everything. Nothing in ArcPro pulls from iNaturalist or uploads a table to AGOL the way you want. Because you built those as script tools, a model can treat them like any other tool and chain them with Esri's.

We will chain: **FetchINatObservations** (yours) → **Buffer** (Esri's) → **UploadFeatureClassToAGOL** (yours).

1. Right click `PythonIsCool.atbx`, then New > Model.
2. Drag **FetchINatObservations** and **UploadFeatureClassToAGOL** from the Catalog pane into the model. Search for the **Buffer** tool in the Geoprocessing pane and drag that in too.
3. Double click FetchINatObservations. Enter `Odocoileus hemionus` for the scientific name and name the output `deer_points`. Click OK.
4. Double click Buffer. For **Input Features**, choose the output of FetchINatObservations from the dropdown (an arrow appears between the boxes). Set the distance to 1000 Meters and name the output `deer_buffers`. Click OK.
5. Double click UploadFeatureClassToAGOL. For **Input feature class**, choose the output of Buffer. Enter an item title. Click OK.
6. Click **Run**. One run fetches the data, buffers it, and uploads it to AGOL. The Item ID appears in the messages.

Note: the upload tool sends the attribute table, so the buffer shapes themselves stay in your geodatabase.

**Reusing the model:** right click the scientific name and the title in the model and choose **Parameter**. Then double click the model in your toolbox. Those two values now appear as inputs, so you can run it for a different species without opening the model.